# Lesson 4 — Cleaning messy data
**Goal:** fix a real-world messy export: duplicates, missing values, mixed date formats, prices stored as text, inconsistent branch names.

> Data cleaning is ~80% of a data job. This lesson pays the rent.

In [1]:
import pandas as pd
raw = pd.read_csv('data/orders_messy.csv')
raw.head(8)

,order_id,order_date,customer_id,product_id,quantity,branch,channel,payment_method,unit_price,discount_pct
0,O00147,2025-01-31,C0184,P002,2.0,Kariakoo,Online,Mobile Money,45000,0
1,O00392,2025-03-26,C0024,P002,2.0,Arusha CBD,In-store,Cash,45000,0
2,O00304,2025-03-08,C0103,P006,2.0,Kariakoo,Online,Mobile Money,"TSh 240,000",10
3,O00103,2025-01-22,C0260,P001,2.0,Mwanza Rock City,In-store,Cash,"TSh 25,000",0
4,O00194,2025-02-09,C0207,P010,1.0,Mlimani City,In-store,Cash,55000,0
5,O00016,2025-01-05,C0200,P003,2.0,Mlimani City,Online,Mobile Money,85000,10
6,O00255,2025-02-24,C0167,P005,1.0,Mlimani City,Online,Mobile Money,18000,0
7,O00305,2025-03-08,NaN,P012,NaN,Mlimani City,In-store,Card,12000,15


In [2]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 412 entries, 0 to 411
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        412 non-null    str    
 1   order_date      412 non-null    str    
 2   customer_id     406 non-null    str    
 3   product_id      412 non-null    str    
 4   quantity        394 non-null    float64
 5   branch          412 non-null    str    
 6   channel         412 non-null    str    
 7   payment_method  387 non-null    str    
 8   unit_price      412 non-null    str    
 9   discount_pct    412 non-null    int64  
dtypes: float64(1), int64(1), str(8)
memory usage: 32.3 KB


## 1. Duplicates

In [3]:
print('duplicate rows:', raw.duplicated().sum())
df = raw.drop_duplicates().copy()
df.shape

duplicate rows: 12


(400, 10)

## 2. Messy text: branch names

In [4]:
df['branch'].value_counts()

branch
Kariakoo               135
Mlimani City           101
Arusha CBD              62
Mwanza Rock City        48
KARIAKOO                 7
kariakoo                 6
  Mwanza Rock City       6
  Mlimani City           6
MWANZA ROCK CITY         4
MLIMANI CITY             4
Mlimani  City            4
Arusha  CBD              4
mwanza rock city         3
arusha cbd               3
  Kariakoo               3
  Arusha CBD             2
mlimani city             1
Mwanza  Rock  City       1
Name: count, dtype: int64

In [5]:
df['branch'] = (df['branch']
                .str.strip()                         # remove spaces at the ends
                .str.replace(r'\s+', ' ', regex=True) # collapse double spaces
                .str.title())                        # consistent capitals
df['branch'] = df['branch'].replace({'Arusha Cbd': 'Arusha CBD'})   # title() lowercased 'CBD'
df['branch'].value_counts()

branch
Kariakoo            151
Mlimani City        116
Arusha CBD           71
Mwanza Rock City     62
Name: count, dtype: int64

## 3. Numbers stored as text: `unit_price`

In [6]:
df['unit_price'].sample(6, random_state=1).tolist()

['12000', '45000', '45000', '12000', '650000', '55000']

In [7]:
df['unit_price'] = (df['unit_price'].astype(str)
                    .str.replace('TSh', '', regex=False)
                    .str.replace(',', '', regex=False)
                    .str.strip()
                    .astype(int))
df['unit_price'].describe()

count       400.000000
mean      87977.500000
std      144718.331536
min       12000.000000
25%       18000.000000
50%       45000.000000
75%       85000.000000
max      650000.000000
Name: unit_price, dtype: float64

## 4. Mixed date formats

In [8]:
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', dayfirst=True)
df['order_date'].dt.year.value_counts()

order_date
2025    400
Name: count, dtype: int64

## 5. Missing values — decide per column

In [9]:
df.isna().sum()

order_id           0
order_date         0
customer_id        6
product_id         0
quantity          18
branch             0
channel            0
payment_method    25
unit_price         0
discount_pct       0
dtype: int64

In [10]:
# quantity missing -> we can't know it; drop those rows
# payment_method missing -> keep the sale, label it 'Unknown'
# customer_id missing -> keep, label 'Walk-in'
df = df.dropna(subset=['quantity'])
df['payment_method'] = df['payment_method'].fillna('Unknown')
df['customer_id'] = df['customer_id'].fillna('Walk-in')
df['quantity'] = df['quantity'].astype(int)
df.isna().sum().sum()

np.int64(0)

In [11]:
print(f'{len(raw)} raw rows -> {len(df)} clean rows')
df.to_csv('data/orders_clean_sample.csv', index=False)
df.dtypes

412 raw rows -> 382 clean rows


order_id                     str
order_date        datetime64[us]
customer_id                  str
product_id                   str
quantity                   int64
branch                       str
channel                      str
payment_method               str
unit_price                 int64
discount_pct               int64
dtype: object

## Key takeaways
- Order of work: duplicates → text → types → dates → missing values.
- `.str` methods clean text; `pd.to_datetime(format='mixed', dayfirst=True)` rescues mixed dates.
- Missing values: drop, fill, or label — **decide per column and write down why**.

**Exercise:** how many rows had a price stored as text? (Hint: check before converting.)